# Fetal Health Multiclassification Using Machine Learning

**Rujul Jain — PES1UG24CS388**  
**Sanjana S Aithal — PES1UG24CS421**

## Objective
Classify CTG records into Normal, Suspect, and Pathological fetal-state classes.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
ROOT = Path.cwd().resolve()
if not (ROOT/'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from download_dataset import download_dataset
from preprocessing import prepare
from models import get_models


In [ ]:
DATA = ROOT/'data'/'fetal_health.csv'
if not DATA.exists(): download_dataset()
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())
display(df.describe().T)

## Class Distribution

In [ ]:
counts=df['NSP'].value_counts().sort_index()
display(counts)
sns.barplot(x=['Normal','Suspect','Pathological'], y=counts.values, color='black')
plt.ylabel('Number of records'); plt.xlabel('Fetal health class'); plt.show()

## Correlation Analysis

In [ ]:
plt.figure(figsize=(12,9))
sns.heatmap(df.corr(numeric_only=True), cmap='Greys', center=0)
plt.title('Feature Correlation Heatmap'); plt.show()

## Train-Test Split and Scaling

A stratified 70/30 split is used. The scaler is fitted only on training data.

In [ ]:
X_train,X_test,X_train_scaled,X_test_scaled,y_train,y_test=prepare(df)
print(X_train.shape, X_test.shape)

## Model Training and Evaluation

In [ ]:
models=get_models()
scaled={'Logistic Regression','Linear SVM','K-Nearest Neighbors','Support Vector Machine','Multi-layer Perceptron'}
rows=[]
for name,model in models.items():
    Xtr=X_train_scaled if name in scaled else X_train
    Xte=X_test_scaled if name in scaled else X_test
    model.fit(Xtr,y_train)
    pred=model.predict(Xte)
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Macro Precision':precision_score(y_test,pred,average='macro',zero_division=0),'Macro Recall':recall_score(y_test,pred,average='macro',zero_division=0),'Macro F1':f1_score(y_test,pred,average='macro',zero_division=0),'Weighted F1':f1_score(y_test,pred,average='weighted',zero_division=0)})
results=pd.DataFrame(rows).sort_values('Macro F1',ascending=False)
display(results)

## Final Model Confusion Matrix

In [ ]:
final_name=results.iloc[0]['Model']
model=models[final_name]
Xte=X_test_scaled if final_name in scaled else X_test
pred=model.predict(Xte)
cm=confusion_matrix(y_test,pred,labels=[1,2,3])
sns.heatmap(cm,annot=True,fmt='d',cmap='Greys',cbar=False,xticklabels=['Normal','Suspect','Pathological'],yticklabels=['Normal','Suspect','Pathological'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion Matrix - {final_name}'); plt.show()
print(classification_report(y_test,pred,target_names=['Normal','Suspect','Pathological'],zero_division=0))

## Conclusion
Use the actual generated metrics and confusion matrix when writing the final discussion. Do not copy numerical results from the Stanford reference paper.